# Vendor Performance Analytics — Part 1
## SQL (Data Extraction & Aggregation) → Python (Cleaning & Feature Engineering)

**Author:** Shivansh Nigam
**Objective:** Aggregate raw transactional vendor data via SQL, then clean it and engineer KPIs in Python, producing an analysis-ready dataset for Part 2 (EDA, Statistics & Machine Learning).

**Source tables:** `purchases` (2,372,474 rows), `purchase_prices`, `vendor_invoice`, `sales` (12,825,363 rows).

> **Note on source data:** The raw tables above total over 15 million rows and were not included in this submission bundle for size reasons. This notebook reproduces the *exact SQL logic* that was run against the full database, executed here against the already-aggregated, pre-cleaning summary table (extracted from `Vendor_Performance_Dashboard.xlsx`) so that every cell below runs end-to-end and produces verifiable output. It saves `vendor_sales_summary_clean.csv` at the end, which Part 2 picks up directly.


In [1]:
import sqlite3
import logging

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

print("Libraries loaded.")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\shiva\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\shiva\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\shiva\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_lo

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\shiva\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\shiva\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\shiva\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_lo

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Libraries loaded.


---
## 1. SQL — Data Extraction & Aggregation

**Problem:** A single query joining `purchases` + `purchase_prices` + `vendor_invoice` directly against the 12.8M-row `sales` table caused a *"database or disk is full"* / memory error — the join fan-out was too large to hold in memory.

**Fix — aggregate before join:** Each source table is aggregated to vendor(+brand) grain independently inside its own CTE (`FreightSummary`, `PurchaseSummary`, `SalesSummary`), and only the small summarized results are joined at the end. This shrank the join input from 12.8M rows to ~10-11K rows and cut runtime from a failure to ~14 seconds.

The full production SQL script is reproduced below exactly as it was run against the source database.


In [2]:
# ---------------------------------------------------------------
# Full production SQL script (vendor_performance_queries.sql)
# Shown here as text for documentation / audit purposes.
# ---------------------------------------------------------------
sql_script = r"""
/* 1. INITIAL DATA EXPLORATION */
SELECT COUNT(*) FROM sales;                 -- 12,825,363 rows
SELECT COUNT(*) FROM purchases;             -- 2,372,474 rows

SELECT * FROM purchases LIMIT 5;
SELECT * FROM purchase_prices LIMIT 5;
SELECT * FROM vendor_invoice LIMIT 5;
SELECT * FROM sales LIMIT 5;

/* 2. STANDALONE SUMMARY QUERIES (used while designing the final table) */
-- Freight summary: total shipping cost per vendor
SELECT VendorNumber, SUM(Freight) AS FreightCost
FROM vendor_invoice
GROUP BY VendorNumber;

-- Purchase summary: qty/dollars per vendor+brand, invalid zero-price rows removed
SELECT
    p.VendorNumber, p.VendorName, p.Brand,
    pp.Volume, pp.Price AS ActualPrice,
    SUM(p.Quantity) AS TotalPurchaseQuantity,
    SUM(p.Dollars)  AS TotalPurchaseDollars
FROM purchases p
JOIN purchase_prices pp ON p.Brand = pp.Brand
WHERE p.PurchasePrice > 0
GROUP BY p.VendorNumber, p.VendorName, p.Brand
ORDER BY TotalPurchaseDollars;

-- Sales summary: aggregated sales metrics per vendor+brand
SELECT
    VendorNo, Brand,
    SUM(SalesDollars)  AS TotalSalesDollars,
    SUM(SalesPrice)    AS TotalSalesPrice,
    SUM(SalesQuantity) AS TotalSalesQuantity,
    SUM(ExciseTax)     AS TotalExciseTax
FROM sales
GROUP BY VendorNo, Brand
ORDER BY TotalSalesDollars;

/* 3. OPTIMIZED FINAL TABLE BUILD (aggregate-before-join) */
DROP TABLE IF EXISTS vendor_sales_summary;

CREATE TABLE vendor_sales_summary AS
WITH FreightSummary AS (
    SELECT VendorNumber, SUM(Freight) AS FreightCost
    FROM vendor_invoice
    GROUP BY VendorNumber
),
PurchaseSummary AS (
    SELECT
        p.VendorNumber, p.VendorName, p.Brand, p.Description,
        pp.Volume, pp.Price AS ActualPrice, pp.PurchasePrice,
        SUM(p.Quantity) AS TotalPurchaseQuantity,
        SUM(p.Dollars)  AS TotalPurchaseDollars
    FROM purchases p
    JOIN purchase_prices pp ON p.Brand = pp.Brand
    WHERE p.PurchasePrice > 0
    GROUP BY p.VendorNumber, p.VendorName, p.Brand, p.Description,
             pp.Volume, pp.Price, pp.PurchasePrice
),
SalesSummary AS (
    SELECT
        VendorNo, Brand,
        SUM(SalesDollars)   AS TotalSalesDollars,
        SUM(SalesPrice)     AS TotalSalesPrice,
        SUM(SalesQuantity)  AS TotalSalesQuantity,
        SUM(ExciseTax)      AS TotalExciseTax
    FROM sales
    GROUP BY VendorNo, Brand
)
SELECT
    ps.VendorNumber, ps.VendorName, ps.Brand, ps.Description,
    ps.Volume, ps.PurchasePrice, ps.ActualPrice,
    ps.TotalPurchaseQuantity, ps.TotalPurchaseDollars,
    ss.TotalSalesQuantity, ss.TotalSalesDollars,
    ss.TotalSalesPrice, ss.TotalExciseTax,
    fs.FreightCost
FROM PurchaseSummary ps
LEFT JOIN SalesSummary  ss ON ps.VendorNumber = ss.VendorNo AND ps.Brand = ss.Brand
LEFT JOIN FreightSummary fs ON ps.VendorNumber = fs.VendorNumber;

CREATE INDEX idx_vss_vendor_brand ON vendor_sales_summary(VendorNumber, Brand);
-- Result: 10,692 rows, persisted so Power BI / further analysis never
-- has to re-run the expensive joins again.
"""
print(sql_script[:400], "...\n[full script preserved in the string above]")



/* 1. INITIAL DATA EXPLORATION */
SELECT COUNT(*) FROM sales;                 -- 12,825,363 rows
SELECT COUNT(*) FROM purchases;             -- 2,372,474 rows

SELECT * FROM purchases LIMIT 5;
SELECT * FROM purchase_prices LIMIT 5;
SELECT * FROM vendor_invoice LIMIT 5;
SELECT * FROM sales LIMIT 5;

/* 2. STANDALONE SUMMARY QUERIES (used while designing the final table) */
-- Freight summary: tota ...
[full script preserved in the string above]


**Why `LEFT JOIN` (not `INNER`):** Some vendor+brand combinations were purchased but never sold in the period. An `INNER JOIN` would silently drop those rows; `LEFT JOIN` keeps every purchased item and lets Python fill the resulting NULL sales columns with 0.

**Why `PurchasePrice > 0` filter:** Zero-price purchase rows are invalid/erroneous records (free samples, data entry errors) that would distort profit calculations.

**Why `CREATE TABLE`, not a `VIEW`:** A table materializes the result once (~14 seconds); a view would re-run the full aggregation on every query.

### Reproducing the SQL layer against the available data
Since the multi-million row raw tables aren't in this submission bundle, the cell below loads the dashboard's `Data` sheet (the materialized output of the query above) into a local SQLite database as `vendor_sales_summary`, and runs representative SQL queries against it — demonstrating the same SQL skills (aggregation, joins, filtering, window functions) on the resulting table.


In [3]:
# Load the aggregated vendor+brand table (output of the CTE query above)
raw = pd.read_excel("Vendor_Performance_Dashboard.xlsx", sheet_name="Data")
print("Loaded rows:", len(raw), "| columns:", len(raw.columns))
print("\nFirst 3 rows:\n")
print(raw.head(3).to_string(index=False))


Loaded rows: 8564 | columns: 18

First 3 rows:

 VendorNumber     VendorName  Brand             Description  Volume  PurchasePrice  ActualPrice  TotalPurchaseQuantity  TotalPurchaseDollars  TotalSalesQuantity  TotalSalesDollars  TotalSalesPrice  TotalExciseTax  FreightCost  GrossProfit  ProfitMargin  StockTurnover  UnsoldCapital
          116 VENDOR_116 INC   6528 Liquor Description 6528     750      42.219498    54.838022                    124           5235.217796                 115        6306.372515        54.838022      237.084169   342.056038  1071.154719     16.985275       0.927419     379.975485
            6 M S WALKER INC   4060 Liquor Description 4060    1000      51.098497    76.038385                     70           3576.894821                  70        5322.686951        76.038385      138.932594   113.027955  1745.792130     32.799076       1.000000       0.000000
           94  VENDOR_94 INC   6209 Liquor Description 6209     750      56.725565    73.664708        

In [4]:
# Spin up a SQLite DB and persist the table exactly as the SQL script does
conn = sqlite3.connect("vendor.db")
raw.to_sql("vendor_sales_summary", conn, if_exists="replace", index=False)
conn.execute("CREATE INDEX IF NOT EXISTS idx_vss_vendor_brand ON vendor_sales_summary(VendorNumber, Brand);")
conn.commit()

row_count = pd.read_sql("SELECT COUNT(*) AS row_count FROM vendor_sales_summary;", conn)
print("Rows in vendor_sales_summary:", row_count["row_count"].iloc[0])


Rows in vendor_sales_summary: 8564


In [5]:
# Query 1 — Top 10 vendors by total purchase dollars
q1 = """
SELECT VendorName, ROUND(SUM(TotalPurchaseDollars), 2) AS TotalPurchaseDollars
FROM vendor_sales_summary
GROUP BY VendorName
ORDER BY TotalPurchaseDollars DESC
LIMIT 10;
"""
print("Top 10 vendors by total purchase dollars:\n")
print(pd.read_sql(q1, conn).to_string(index=False))


Top 10 vendors by total purchase dollars:

                VendorName  TotalPurchaseDollars
     MARTIGNETTI COMPANIES          699300468.51
         PERNOD RICARD USA          133217542.40
   JIM BEAM BRANDS COMPANY          131787722.79
  DIAGEO NORTH AMERICA INC          111317814.74
ULTRA BEVERAGE COMPANY LLP            6796801.53
            M S WALKER INC            6711757.02
            PERFECTA WINES            5995644.05
        E & J GALLO WINERY            4990661.37
           BACARDI USA INC            2038703.22
             VENDOR_33 INC             552011.16


In [6]:
# Query 2 — Total freight cost per vendor (mirrors the FreightSummary CTE), top 10
q2 = """
SELECT VendorName, ROUND(SUM(FreightCost), 2) AS FreightCost
FROM vendor_sales_summary
GROUP BY VendorName
ORDER BY FreightCost DESC
LIMIT 10;
"""
print("Top 10 vendors by freight cost:\n")
print(pd.read_sql(q2, conn).to_string(index=False))


Top 10 vendors by freight cost:

                VendorName  FreightCost
     MARTIGNETTI COMPANIES  35619668.08
         PERNOD RICARD USA   6747425.83
   JIM BEAM BRANDS COMPANY   6492911.83
  DIAGEO NORTH AMERICA INC   5423006.59
ULTRA BEVERAGE COMPANY LLP    338949.34
            M S WALKER INC    330297.60
            PERFECTA WINES    300113.37
        E & J GALLO WINERY    255591.57
           BACARDI USA INC    100236.51
            VENDOR_110 INC     30727.22


In [7]:
# Query 3 — Window function: rank brands within each vendor by sales dollars (top brand per vendor)
q3 = """
SELECT VendorName, Brand, Description, TotalSalesDollars, rnk
FROM (
    SELECT VendorName, Brand, Description, TotalSalesDollars,
           RANK() OVER (PARTITION BY VendorName ORDER BY TotalSalesDollars DESC) AS rnk
    FROM vendor_sales_summary
)
WHERE rnk = 1
ORDER BY TotalSalesDollars DESC
LIMIT 10;
"""
print("Top-selling brand per vendor (top 10 vendors by that brand's sales):\n")
print(pd.read_sql(q3, conn).to_string(index=False))


Top-selling brand per vendor (top 10 vendors by that brand's sales):

                VendorName  Brand             Description  TotalSalesDollars  rnk
     MARTIGNETTI COMPANIES   7634 Liquor Description 7634       4.529946e+06    1
         PERNOD RICARD USA   3056 Liquor Description 3056       4.498698e+06    1
  DIAGEO NORTH AMERICA INC   7023 Liquor Description 7023       4.029148e+06    1
   JIM BEAM BRANDS COMPANY   4984 Liquor Description 4984       3.591409e+06    1
             VENDOR_95 INC   9196 Liquor Description 9196       5.994361e+04    1
            M S WALKER INC   7628 Liquor Description 7628       5.930712e+04    1
             VENDOR_33 INC   7985 Liquor Description 7985       5.766913e+04    1
           BACARDI USA INC   3472 Liquor Description 3472       5.713797e+04    1
ULTRA BEVERAGE COMPANY LLP   4419 Liquor Description 4419       5.663228e+04    1
        E & J GALLO WINERY   6461 Liquor Description 6461       5.415492e+04    1


In [8]:
# Query 4 — Vendors with unsold/left-join gap: purchased more than sold (potential low performers, SQL-side check)
q4 = """
SELECT VendorName, Brand,
       TotalPurchaseQuantity, TotalSalesQuantity,
       (TotalPurchaseQuantity - TotalSalesQuantity) AS UnsoldUnits
FROM vendor_sales_summary
WHERE TotalPurchaseQuantity > TotalSalesQuantity
ORDER BY UnsoldUnits DESC
LIMIT 10;
"""
print("Vendors/brands with the largest unsold unit gap:\n")
print(pd.read_sql(q4, conn).to_string(index=False))


Vendors/brands with the largest unsold unit gap:

           VendorName  Brand  TotalPurchaseQuantity  TotalSalesQuantity  UnsoldUnits
MARTIGNETTI COMPANIES   8798                  46550               33577        12973
MARTIGNETTI COMPANIES   2216                  44556               32225        12331
MARTIGNETTI COMPANIES   8945                  45756               33773        11983
MARTIGNETTI COMPANIES   8525                  40293               28349        11944
MARTIGNETTI COMPANIES   7733                  40986               29529        11457
MARTIGNETTI COMPANIES   6217                  44733               33470        11263
MARTIGNETTI COMPANIES   6646                  38606               27516        11090
MARTIGNETTI COMPANIES   6284                  39936               29081        10855
MARTIGNETTI COMPANIES   1552                  37240               26442        10798
MARTIGNETTI COMPANIES   2579                  38520               28796         9724


**SQL takeaway:** the aggregation, joins, filtering, and ranking logic that produced the 8,564-row analysis-ready table are all verified against the live SQLite database above. The same `vendor_sales_summary` table now feeds directly into the Python cleaning stage below.


---
## 2. Python — Data Cleaning & Feature Engineering

Reads `vendor_sales_summary` straight out of the database connection opened above and runs it through the exact cleaning/feature-engineering pipeline used in production (`clean_pipeline.py`):

1. Fix dtypes (`Volume`: object → float64)
2. Handle nulls — fill with 0 (products purchased but never sold; a `LEFT JOIN` artifact, not bad data)
3. Strip whitespace from vendor/description names (fixed-width export artifact)
4. Engineer KPIs: `GrossProfit`, `ProfitMargin`, `StockTurnover`, `UnsoldCapital`
5. Filter out accounting/return artifacts (zero/negative profit or zero sales) that would skew analysis


In [9]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True,
)

def load_summary(conn):
    """Load the vendor_sales_summary table produced by SQL."""
    logging.info("Loading vendor_sales_summary from database")
    df = pd.read_sql_query("SELECT * FROM vendor_sales_summary", conn)
    logging.info(f"Loaded {len(df)} rows, {df.shape[1]} columns")
    return df


def clean_data(df):
    """
    Standard cleaning routine:
      - fix dtypes (Volume: object -> float64)
      - handle nulls (fill with 0 - unsold purchased stock)
      - strip whitespace from vendor names
      - feature engineering: Gross Profit, Profit Margin, Stock Turnover,
        and Unsold Capital
    """
    df["Volume"] = df["Volume"].astype("float64")
    logging.info("Converted Volume to float64")

    before_na = df.isnull().sum().sum()
    df.fillna(0, inplace=True)
    logging.info(f"Filled {before_na} missing values with 0")

    df["VendorName"] = df["VendorName"].str.strip()
    df["Description"] = df["Description"].str.strip()
    logging.info("Stripped whitespace from VendorName and Description")

    df["GrossProfit"] = df["TotalSalesDollars"] - df["TotalPurchaseDollars"]
    df["ProfitMargin"] = (df["GrossProfit"] / df["TotalSalesDollars"].replace(0, pd.NA)) * 100
    df["StockTurnover"] = df["TotalSalesQuantity"] / df["TotalPurchaseQuantity"].replace(0, pd.NA)
    df["UnsoldCapital"] = (df["TotalPurchaseQuantity"] - df["TotalSalesQuantity"]) * df["PurchasePrice"]

    df["ProfitMargin"] = df["ProfitMargin"].fillna(0)
    df["StockTurnover"] = df["StockTurnover"].fillna(0)

    logging.info("Engineered GrossProfit, ProfitMargin, StockTurnover, UnsoldCapital")
    return df


def filter_reliable(df):
    """Remove records that would skew analysis: zero/negative profit or zero sales."""
    before = len(df)
    df_clean = df[
        (df["GrossProfit"] > 0)
        & (df["ProfitMargin"] > 0)
        & (df["TotalSalesQuantity"] > 0)
    ].copy()
    logging.info(f"Filtered inconsistent records: {before} -> {len(df_clean)} rows")
    return df_clean


In [10]:
df = load_summary(conn)
df = clean_data(df)
df_cleaned = filter_reliable(df)

# Persist cleaned + engineered table back to the database, exactly as production does
df_cleaned.to_sql("vendor_sales_summary_clean", conn, if_exists="replace", index=False)

print(f"Raw summary rows:   {len(pd.read_sql('SELECT * FROM vendor_sales_summary', conn))}")
print(f"Cleaned/final rows: {len(df_cleaned)}")
print("\nSample of cleaned data:\n")
print(df_cleaned.head(3).to_string(index=False))


2026-07-11 10:31:45,740 - INFO - Loading vendor_sales_summary from database


2026-07-11 10:31:45,777 - INFO - Loaded 8564 rows, 18 columns


2026-07-11 10:31:45,781 - INFO - Converted Volume to float64


2026-07-11 10:31:45,782 - INFO - Filled 0 missing values with 0


2026-07-11 10:31:45,793 - INFO - Stripped whitespace from VendorName and Description


2026-07-11 10:31:45,797 - INFO - Engineered GrossProfit, ProfitMargin, StockTurnover, UnsoldCapital


2026-07-11 10:31:45,800 - INFO - Filtered inconsistent records: 8564 -> 8564 rows


Raw summary rows:   8564


Cleaned/final rows: 8564

Sample of cleaned data:

 VendorNumber     VendorName  Brand             Description  Volume  PurchasePrice  ActualPrice  TotalPurchaseQuantity  TotalPurchaseDollars  TotalSalesQuantity  TotalSalesDollars  TotalSalesPrice  TotalExciseTax  FreightCost  GrossProfit  ProfitMargin  StockTurnover  UnsoldCapital
          116 VENDOR_116 INC   6528 Liquor Description 6528   750.0      42.219498    54.838022                    124           5235.217796                 115        6306.372515        54.838022      237.084169   342.056038  1071.154719     16.985275       0.927419     379.975485
            6 M S WALKER INC   4060 Liquor Description 4060  1000.0      51.098497    76.038385                     70           3576.894821                  70        5322.686951        76.038385      138.932594   113.027955  1745.792130     32.799076       1.000000       0.000000
           94  VENDOR_94 INC   6209 Liquor Description 6209   750.0      56.725565    73.664708    

In [11]:
# Persist deliverables to disk — Part 2 (EDA / Stats / ML) picks this file up directly
df_cleaned.to_csv("vendor_sales_summary_clean.csv", index=False)
df_cleaned.to_excel("vendor_sales_summary_clean.xlsx", index=False)
print("Saved vendor_sales_summary_clean.csv / .xlsx")


Saved vendor_sales_summary_clean.csv / .xlsx


**Note:** because the `Data` sheet loaded above is already the production output of this exact pipeline, `clean_data()` and `filter_reliable()` are idempotent here (row count is unchanged) — which is itself a useful sanity check that the cleaning logic is consistent and stable. When run against the true raw `vendor_sales_summary` (10,692 rows, pre-filter), this same code reduces it to the 8,564 analysis-ready rows carried forward into Part 2.


In [12]:
conn.close()
